# AFSP full run

Operational runbook for the one long thread that produces the thesis's val numbers.
It runs on a **fresh A100** session and is organised into **four phases** because the
COMET stack and the generation stack cannot share one Python environment
(`requirements-comet.txt` pins `transformers==4.57.6` / `numpy==1.26.4`, the
generation stack pins `transformers==5.12.1` / `numpy==2.4.1`). The pipeline's own
ordering forces the alternation:

| Phase | Runtime | What runs | Depends on |
|------|---------|-----------|------------|
| 1 | **generation** (`requirements.txt`) | `build_index`, `afsp_sweep` (the ~26k-gen run) | — |
| 2 | **COMET** (`requirements-comet.txt`) | `afsp_verify` (COMET + judge), **freeze** | Phase 1 sweep result |
| 3 | **generation** (`requirements.txt`) | five-condition ladder generation + chrF/BLEU + stylometrics + judge Φ | frozen `(k, λ)` |
| 4 | **COMET** (`requirements-comet.txt`) | ladder COMET + paired bootstrap | Phase 3 ladder outputs |


---
## Phase 1 — generation runtime · the sweep

Fresh A100, default (generation) stack. This is the multi-hour run that gates
everything downstream.

In [1]:
# Confirm the GPU: the sweep regenerates with Qwen2.5-7B in bf16 (~15 GB weights).
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA GeForce RTX 4090, 24564 MiB


In [2]:
import os
if not os.path.isdir('Style-Aware-MT'):
    !git clone --branch feat/afsp-implementation https://github.com/prnamhr/Style-Aware-MT.git
%cd Style-Aware-MT
!git rev-parse --short HEAD

Cloning into 'Style-Aware-MT'...
remote: Enumerating objects: 609, done.
remote: Counting objects: 100% (238/238), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 609 (delta 144), reused 150 (delta 87), pack-reused 371 (from 1)
Receiving objects: 100% (609/609), 8.58 MiB | 14.77 MiB/s, done.
Resolving deltas: 100% (355/355), done.
/workspace/Style-Aware-MT/notebooks/Style-Aware-MT
568ba73


In [3]:
# Generation stack.
!pip install -r requirements.txt

!pip uninstall -y torchvision torchaudio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 1.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 8.6 MB/s eta 0:00:00
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 6.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 19.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 65.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 50.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 68.9 MB/s eta 0:00:00
 

The register centroid (`results/stylometrics_centroid.json`) is committed, so it is
already present. The kNN index is git-ignored and must be rebuilt each session.

In [4]:
import torch; print(torch.__version__, torch.cuda.is_available(), torch.version.cuda)

2.12.0+cu130 True 13.0


In [7]:
import os, getpass
os.environ["HF_HUB_DISABLE_XET"] = "1"
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("HF_TOKEN: ")

In [9]:
!python3 manage.py build_index --config configs/base_qwen.yaml

Embedding 10860 Persian/Arabic training sources with intfloat/multilingual-e5-large-instruct ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda
config_sentence_transformers.json: 100%|███████| 128/128 [00:00<00:00, 510kB/s]
README.md: 100%|████████████████████████████| 140k/140k [00:00<00:00, 12.5MB/s]
sentence_xlm-roberta_config.json: 100%|██████| 53.0/53.0 [00:00<00:00, 236kB/s]
config.json: 100%|████████████████████████████| 690/690 [00:00<00:00, 3.24MB/s]
model.safetensors: 100%|██████████████████| 1.12G/1.12G [00:20<00:00, 54.9MB/s]
Loading weights: 100%|█████████████████████| 391/391 [00:00<00:00, 3984.03it/s]
tokenizer_config.json: 100%|██████████████| 1.18k/1.18k [00:00<00:00, 2.92MB/s]
sentencepiece.bpe.model: 100%|████████████| 5.07M/5.07M [00:00<00:00, 60.8MB/s]
tokenizer.json: 100%|█████████████████████| 17.1M/17.1M [00:00<00:00, 46.2MB/s]
Batches: 100%|███████████████████████████████| 340/340 [00:05<00:00, 61.58it/s]
Wrote index to data/knn_index/ : embedd

In [10]:
!python3 manage.py afsp_sweep --config configs/afsp_sweep.yaml

config.json: 100%|████████████████████████████| 663/663 [00:00<00:00, 2.70MB/s]
tokenizer_config.json: 100%|██████████████| 7.30k/7.30k [00:00<00:00, 12.9MB/s]
vocab.json: 100%|█████████████████████████| 2.78M/2.78M [00:00<00:00, 8.42MB/s]
merges.txt: 100%|█████████████████████████| 1.67M/1.67M [00:00<00:00, 36.9MB/s]
tokenizer.json: 100%|█████████████████████| 7.03M/7.03M [00:00<00:00, 52.5MB/s]
model.safetensors.index.json: 100%|███████| 27.8k/27.8k [00:00<00:00, 39.9MB/s]
Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 4 files:   0%|                                  | 0/4 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0%|       |  0.00B / 3.95GB            
Reconstructing (incomplete total...):   0%|       |  0.00B / 7.81GB            
Reconstructing (incomplete total...):   0%|       |  0.00B / 11.4GB            
Reconstructing (incomplete total...):   0%|       |  0.00B / 15.2GB            
Reconstructing (incomplete total...):  

In [18]:
!git add .
!git commit -m 'feat: new config run and outputs'

On branch feat/afsp-implementation
Your branch is ahead of 'origin/feat/afsp-implementation' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean


In [20]:
import os, getpass
_gh = os.environ.get("GH_TOKEN") or getpass.getpass("GH_TOKEN: ")
!git push https://{_gh}@github.com/prnamhr/Style-Aware-MT.git HEAD:feat/afsp-implementation

In [24]:
# The proxy-picked (k, lambda) the sweep recommends (free/local proxies only).
import json
sweep = json.load(open('results/afsp_sweep_val.json'))
rec = sweep.get('recommended')
print('proxy recommended:', rec and {k: rec[k] for k in ('tag', 'k', 'lambda', 'chrF', 'stylo_dist') if k in rec})

proxy recommended: {'tag': 'afsp_k16_l0.75', 'k': 16, 'lambda': 0.75, 'chrF': 40.05, 'stylo_dist': 0.3643}


---
## Phase 2 — COMET runtime · verify + freeze


In [8]:
%cd /content/Style-Aware-MT
# COMET stack (unbabel-comet + transformers 4.57.6 + numpy 1.26.4). Leaves openai in place
# from Phase 1's install, so the judge pass inside afsp_verify still works.
!pip install -q -r requirements-comet.txt

[Errno 2] No such file or directory: '/content/Style-Aware-MT'
/home/prnamhr/projects/Style-Aware-MT/notebooks/Style-Aware-MT


In [9]:
# The judge is gpt-4.1 (configs/judge_eval.yaml) -> needs an OpenAI key.
import os, getpass
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY: ')

In [33]:
!USE_TF=0 python -m src.infer.afsp_verify --config configs/afsp_sweep.yaml \
    --judge-config configs/judge_eval.yaml --top 3


Confirming top 3 proxy cells from afsp_sweep_val.json (['afsp_k16_l0.75', 'afsp_k8_l0.75', 'afsp_k8_l0.5']) on val
Loading checkpoint shards: 100% 4/4 [00:04<00:00,  1.06s/it]
skip afsp_k16_l0.75: outputs/sweep/afsp_k16_l0.75_val.jsonl exists (use --overwrite to regenerate)
skip afsp_k8_l0.75: outputs/sweep/afsp_k8_l0.75_val.jsonl exists (use --overwrite to regenerate)
skip afsp_k8_l0.5: outputs/sweep/afsp_k8_l0.5_val.jsonl exists (use --overwrite to regenerate)
Fetching 5 files:   0% 0/5 [00:00<?, ?it/s]
README.md: 3.40kB [00:00, 14.8MB/s]

checkpoints/model.ckpt:   0% 0.00/2.32G [00:00<?, ?B/s]

.gitattributes: 1.48kB [00:00, 10.6MB/s]
Fetching 5 files:  20% 1/5 [00:00<00:00,  6.86it/s]

hparams.yaml: 100% 567/567 [00:00<00:00, 6.55MB/s]


LICENSE: 9.69kB [00:00, 41.1MB/s]

checkpoints/model.ckpt:   0% 194k/2.32G [00:01<3:24:16, 190kB/s]
checkpoints/model.ckpt:   6% 134M/2.32G [00:01<00:19, 110MB/s]  
checkpoints/model.ckpt:  12% 268M/2.32G [00:02<00:12, 169MB/s]
checkpoints/model.ck

In [34]:
# The freeze decision from the reported metrics.
import json
v = json.load(open('results/afsp_verify_val.json'))
print('freeze tag :', v['freeze'], '(proxy pick held)' if v['proxy_pick_held'] else '(runner-up overtook the proxy pick)')
for c in v['cells']:
    mark = '  <== freeze' if c['tag'] == v['freeze'] else ''
    print(f"  {c['tag']:<16} k={c['k']} lambda={c['lambda']}  COMET {c['comet_system']:.4f}  Phi {c['judge_mean']}{mark}")

freeze tag : afsp_k8_l0.75 (runner-up overtook the proxy pick)
  afsp_k16_l0.75   k=16 lambda=0.75  COMET 0.6827  Phi 3.45578231292517
  afsp_k8_l0.75    k=8 lambda=0.75  COMET 0.6794  Phi 3.4829931972789114  <== freeze
  afsp_k8_l0.5     k=8 lambda=0.5  COMET 0.6823  Phi 3.4784580498866213


### Freeze `(k, λ)` into `configs/base_qwen.yaml`


In [52]:
import json, re, pathlib
v = json.load(open('results/afsp_verify_val.json'))
k = lam = None
for c in v['cells']:
    if c['tag'] == v['freeze']:
        k, lam = c['k'], c['lambda']
assert k is not None, 'freeze cell not found in results/afsp_verify_val.json'

p = pathlib.Path('configs/base_qwen.yaml')
text = p.read_text(encoding='utf-8')
text = re.sub(r'(?m)^(\s*k:\s*)\S+', lambda m: f'{m.group(1)}{k}', text, count=1)
text = re.sub(r'(?m)^(\s*lambda_style:\s*)\S+', lambda m: f'{m.group(1)}{lam}', text, count=1)
p.write_text(text, encoding='utf-8')
print(f'froze retrieval.k={k}, afsp.lambda_style={lam} into configs/base_qwen.yaml')
!grep -nE '^\s*(k|lambda_style):' configs/base_qwen.yaml

froze retrieval.k=8, afsp.lambda_style=0.75 into configs/base_qwen.yaml
26:  k: 8                                   # shots; sweep {2,4,8} on dev later
33:  lambda_style: 0.75                            # target-distribution-priority weight in [0, 1]


In [10]:
!python manage.py comet \
  --conditions afsp_zeroshot afsp_k8_l0 afsp_k16_l0 afsp_k8_l0.5 afsp_k8_l0.75 afsp_k16_l0.75 \
  --split val --out_dir outputs/sweep
# -> results/comet_val.json  (full + margin + base, same axis)

hparams.yaml: 100%|███████████████████████████| 567/567 [00:00<00:00, 7.90MB/s]

LICENSE: 0.00B [00:00, ?B/s]

README.md: 3.40kB [00:00, 19.8MB/s]A
LICENSE: 9.69kB [00:00, 12.5MB/s]

.gitattributes: 1.48kB [00:00, 5.88MB/s]
Fetching 5 files:  20%|█████▏                    | 1/5 [00:00<00:00,  4.11it/s]
checkpoints/model.ckpt:   0%|                      | 0.00/2.32G [00:00<?, ?B/s]
checkpoints/model.ckpt:   0%|             | 194k/2.32G [00:01<5:02:08, 128kB/s]
checkpoints/model.ckpt:   0%|             | 194k/2.32G [00:19<5:02:08, 128kB/s]
checkpoints/model.ckpt:   6%|▊             | 134M/2.32G [00:43<11:46, 3.10MB/s]
checkpoints/model.ckpt:  11%|█▌            | 256M/2.32G [00:45<05:02, 6.83MB/s]
checkpoints/model.ckpt:  12%|█▌            | 268M/2.32G [00:48<05:25, 6.32MB/s]
checkpoints/model.ckpt:  12%|█▌            | 268M/2.32G [00:59<05:25, 6.32MB/s]
checkpoints/model.ckpt:  17%|██▍           | 402M/2.32G [01:27<07:17, 4.39MB/s]
checkpoints/model.ckpt:  23%|███▏          | 536M/2.32G 

In [11]:
# Judge — API (gpt-4.1). ONLY the not-yet-judged cells (base + margin):

!python manage.py judge \
  --conditions afsp_zeroshot afsp_k8_l0 afsp_k16_l0 \
  --split val --out_dir outputs/sweep \
  --config configs/judge_eval.yaml


Judging 1323 segments for afsp_zeroshot with gpt-4.1 ...
  afsp_zeroshot    Φ 3.510  (coverage 100%)
Judging 1323 segments for afsp_k8_l0 with gpt-4.1 ...
  afsp_k8_l0       Φ 3.519  (coverage 100%)
Judging 1323 segments for afsp_k16_l0 with gpt-4.1 ...
  afsp_k16_l0      Φ 3.489  (coverage 100%)
Judge usage: {'calls': 3969, 'prompt_tokens': 1634766, 'completion_tokens': 224556, 'cost_usd': 5.066}
Wrote results/judge_val.json


---
## Phase 3 — generation runtime · the five-condition ladder


In [13]:
%cd /content/Style-Aware-MT
!pip install -q -r requirements.txt
# Same torchvision/torchaudio ABI mismatch as Phase 1 — drop them (text-only pipeline).
!pip uninstall -y torchvision torchaudio

[Errno 2] No such file or directory: '/content/Style-Aware-MT'
/home/prnamhr/projects/Style-Aware-MT/notebooks/Style-Aware-MT


In [ ]:
# Generate all five rungs on full val (each is resumable via its own outputs/*_val.jsonl).
CONDS = ['zeroshot', 'random_fewshot', 'knn_fewshot', 'afsp_margin', 'afsp_full']
for c in CONDS:
    print(f'\n=== generating {c} ===')
    !python manage.py infer --condition {c} --config configs/base_qwen.yaml

In [ ]:
# Adequacy proxies (chrF/BLEU) and register stylometrics — free/local, no COMET.
CONDS = 'zeroshot random_fewshot knn_fewshot afsp_margin afsp_full'
!python manage.py eval         --conditions {CONDS} --split val
!python manage.py stylometrics --conditions {CONDS} --split val

In [ ]:
# Register fidelity (judge Phi) over the ladder — OpenAI gpt-4.1.
import os, getpass
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY: ')
CONDS = 'zeroshot random_fewshot knn_fewshot afsp_margin afsp_full'
!python manage.py judge --conditions {CONDS} --split val --config configs/judge_eval.yaml

---
## Phase 4 — COMET runtime · ladder COMET + paired bootstrap


In [ ]:
%cd /content/Style-Aware-MT
!pip install -q -r requirements-comet.txt

In [ ]:
CONDS = 'zeroshot random_fewshot knn_fewshot afsp_margin afsp_full'
!python manage.py comet --conditions {CONDS} --split val

In [ ]:
CONDS = 'zeroshot random_fewshot knn_fewshot afsp_margin afsp_full'
!python manage.py bootstrap --metric comet --conditions {CONDS} --split val --adjacent